In [1]:
import pandas as pd
import gzip
from pathlib import Path
import re

# =========================
# ΡΥΘΜΙΣΕΙΣ
# =========================

INPUT_DIR = Path("E:/IDEAL/auxiliarydata/hourly_readings")
OUTPUT_DIR = Path("E:/IDEAL/processed/hourly_per_home")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXCLUDED_HOMES = {"home223"}

# regex patterns
ELECTRIC_PATTERN = re.compile(r"(home\d+).*electric-mains_electric-combined\.csv\.gz")
ROOM_PATTERN = re.compile(r"(home\d+).*room_temperature\.csv\.gz")

# =========================
# ΒΟΗΘΗΤΙΚΕΣ ΣΥΝΑΡΤΗΣΕΙΣ
# =========================

def read_hourly_csv_gz(path):
    """
    Διαβάζει csv.gz χωρίς headers.
    Υποθέτει:
    - στήλη 0: timestamp
    - στήλη 1: value
    """
    return pd.read_csv(
        path,
        header=None,
        names=["timestamp", "value"],
        parse_dates=["timestamp"]
    )


# =========================
# ΚΥΡΙΑ ΛΟΓΙΚΗ
# =========================

# 1. Εντοπισμός όλων των αρχείων
electric_files = []
room_files = {}

for file in INPUT_DIR.glob("*.csv.gz"):
    name = file.name

    m_e = ELECTRIC_PATTERN.match(name)
    if m_e:
        home = m_e.group(1)
        if home not in EXCLUDED_HOMES:
            electric_files.append((home, file))
        continue

    m_r = ROOM_PATTERN.match(name)
    if m_r:
        home = m_r.group(1)
        if home not in EXCLUDED_HOMES:
            room_files.setdefault(home, []).append(file)

# 2. Επεξεργασία ανά σπίτι
for home, electric_path in electric_files:
    print(f"Επεξεργασία {home}")

    # -------------------------
    # Κατανάλωση
    # -------------------------
    df_elec = read_hourly_csv_gz(electric_path)
    df_elec = df_elec.rename(columns={"value": "consumption_Wh"})

    # -------------------------
    # Room temperatures
    # -------------------------
    room_dfs = []

    for room_path in room_files.get(home, []):
        df_room = read_hourly_csv_gz(room_path)
        room_dfs.append(df_room)

    if room_dfs:
        df_rooms_all = pd.concat(room_dfs, ignore_index=True)

        # μέσος όρος ανά timestamp
        df_internal = (
            df_rooms_all
            .groupby("timestamp", as_index=False)["value"]
            .mean()
            .rename(columns={"value": "internal_temperature"})
        )

        # μετατροπή από tenths °C → °C
        df_internal["internal_temperature"] = df_internal["internal_temperature"] / 10.0
    else:
        df_internal = pd.DataFrame(columns=["timestamp", "internal_temperature"])

    # -------------------------
    # Συνένωση
    # -------------------------
    df_final = df_elec.merge(
        df_internal,
        on="timestamp",
        how="left"
    )

    # external temperature (placeholder)
    df_final["external_temperature"] = pd.NA

    # -------------------------
    # Αποθήκευση
    # -------------------------
    out_path = OUTPUT_DIR / f"{home}_hourly.csv"
    df_final.to_csv(out_path, index=False)

print("Ολοκληρώθηκε.")


Επεξεργασία home100
Επεξεργασία home101
Επεξεργασία home102
Επεξεργασία home105
Επεξεργασία home106
Επεξεργασία home107
Επεξεργασία home109
Επεξεργασία home110
Επεξεργασία home113
Επεξεργασία home114
Επεξεργασία home115
Επεξεργασία home116
Επεξεργασία home117
Επεξεργασία home118
Επεξεργασία home119
Επεξεργασία home120
Επεξεργασία home121
Επεξεργασία home122
Επεξεργασία home123
Επεξεργασία home124
Επεξεργασία home125
Επεξεργασία home126
Επεξεργασία home128
Επεξεργασία home129
Επεξεργασία home133
Επεξεργασία home134
Επεξεργασία home135
Επεξεργασία home136
Επεξεργασία home137
Επεξεργασία home138
Επεξεργασία home139
Επεξεργασία home140
Επεξεργασία home141
Επεξεργασία home143
Επεξεργασία home144
Επεξεργασία home145
Επεξεργασία home146
Επεξεργασία home147
Επεξεργασία home148
Επεξεργασία home149
Επεξεργασία home150
Επεξεργασία home151
Επεξεργασία home152
Επεξεργασία home153
Επεξεργασία home154
Επεξεργασία home155
Επεξεργασία home156
Επεξεργασία home157
Επεξεργασία home158
Επεξεργασία home159
